## Objectives

In this lab the class will implement the Metropolis-Hastings algorithm and compare to Gibbs sampling and least-squares estimators. The goal is to provide some familiarity with the implementation of MCMC.

## Generating the data

This is intended to be a very simple dataset with a simple model for the purposes of demonstrating how to set up a Bayesian analysis.


In [ ]:
#Loading libraries
req_packages<-c("MASS", "DescTools")

for(i in c(1:length(req_packages))){
  if (!require(req_packages[i], character.only = TRUE)){
   install.packages(req_packages[i])
  }
}
library(MASS)
library(DescTools)

In [1]:
#I'm cutting and pasting code so it is set up as a series of functions - not very intuitive but a quick adaptation of old code
setX<-function(){
  X=matrix(0,90,3)
  X[1:30,1]=1
  X[31:60,2]=1
  X[61:90,3]=1
  return(X)
}
setY<-function(){
  #simulating 90 data points
  y=rep(0,90)
  #Data is split between 3 treatments with effects (130,220,90)
  y[1:30]=rnorm(30,130,15)
  y[31:60]=rnorm(30,220,15)
  y[61:90]=rnorm(30,90,15)
  return(y)
}
#Generating the incidence matrix X
X=setX()
#generating Y values
y=setY()


## Solving via OLS

First we will obtain solutions for OLS


In [2]:
# solve LSM

beta=solve(t(X)%*%X)%*%t(X)%*%y
vare=(t(y-X%*%solve(t(X)%*%X)%*%t(X)%*%y)%*%(y-X%*%solve(t(X)%*%X)%*%t(X)%*%y))/(length(y)-3)


In [ ]:
print("LSM")
print(beta)
print("Variance Estimate")
print(vare)


## Solving via Bayesian Methods


### Metropolis Hastings Algorithm

Here we use the Metropolis Hastings algorithm to set up the Markov Chain. The functions for implementing the Metropolis Hastings implementation are:

solveBayesMetropolis() - this function initializes the variables for beta and the residual variance and sets the parameters for the proposal distributions.

solveBayesMetroplis() calls two functions:
sampleBetaMetropolis() and sampleSigeMetropolis() - These functions sample new values from the proposal distributions, calculate the acceptance probability and determine whether or not to accept the proposed value. These functions keep sampling values until a new proposed value is accepted. Once a value is accepted it is returned to the main function solveBayesMetropolis()

In [ ]:
solveBayesMetropolis<-function(nIter,X,y,initialBeta,initialSige){

  #initializing matrix to store beta results
  #col 1 = beta1, col2 = beta2, col3 = beta3, col4 = error variance
  samples=matrix(0,nIter,(length(X[1,])+1))

  #Initialize solutions
  betaMu=initialBeta
  betaCurrent=initialBeta
  sigeCurrent=initialSige


  for(i in c(1:nIter)){
    #sample Using MH
    #print(i)

    #sample new value of the residual variance using M-H
    sigeNew=sampleSigeMetropolis(sigeCurrent,y,X,betaCurrent)


    if(is.na(sigeNew)){
      print("Those starting values are terrible.")
      break
    }

    #variance for the full conditional distribution of beta
    V=diag(sigeNew,length(y))
    #variance for the proposal distribution for beta
    Vp=diag(sigeNew,3)

    #sample new values of beta using M-H
    betaNew=sampleBetaMetropolis(betaCurrent,betaMu,y,X,Vp,V)

    if(is.na(betaNew[1])){
      print("Those starting values are terrible.")
      break
    }

    #update solutions
    samples[i,1:3]=betaNew
    betaCurrent=betaNew
    betaMu=c(mean(samples[1:i,1]),mean(samples[1:i,2]),mean(samples[1:i,3]))
    samples[i,4]=sigeNew
    sigeCurrent=sigeNew
  }
  #return smaples
  return(samples)
}

#############################################################
# Function to sample beta values using Metropolis Hastings  #
#############################################################

sampleBetaMetropolis<-function(betaCurrent,betaMu,y,X,Vp,V){

  #using a normal distribution for proposal
  #calculate residuals using the current value for beta
  res=y-X%*%betaCurrent
  #putting the current values for beta into the fully conditional distributions
  pCurrentConditional=det(V)**-.5*exp(-.5*t(res)%*%solve(V)%*%res)
  #calculating the difference between the current values of beta and the mean of the proposal distribution
  rescp=betaCurrent-betaMu
  #putting the current values of beta into the density of the proposal distribution
  pCurrentProposal=det(Vp)**-.5*exp(-.5*t(rescp)%*%solve(Vp)%*%rescp)
  i=0

  #This loop will keep sampling new values until one is accepted
   while(i<1){
    #sample from proposal
    betaProposed=mvrnorm(1,betaMu,Vp)

    #calculate MH ratio
    #calculate residuals using the proposed value for beta
    res=y-X%*%betaProposed
    #putting the proposed values for beta into the fully conditional distributions
    pProposedConditional=det(V)**-.5*exp(-.5*t(res)%*%solve(V)%*%res)
    #calculating the difference between the proposed values of beta and the mean of the proposal distribution
    ressp=betaProposed-betaMu
    #putting the proposed values of beta into the density of the proposal distribution
    pProposedProposal=det(Vp)**-.5*exp(-.5*t(ressp)%*%solve(Vp)%*%ressp)
    #calculating Metropolis Hastings ratio


    MHr= #your code here



    #if the denominator is very small (zero) the ratio is NA
    if(!is.na(MHr)){
       #if the ratio is greater than 1 we automatically accept
       if(MHr>=1){
        betaSampled=betaProposed
        i=1
      } else {
        #If the ratio is less than one we sample a random uniform variable between 0 and 1
        randomNumber=runif(1,0,1)
        #if the sampled number is less than the ratio we accept
        if(randomNumber<=MHr){
          betaSampled=betaProposed
          i=1
        }
      }
    } else {
      #betaSampled=c(NA,NA,NA)
      betaSampled=betaProposed
      i=1
    }
  }

  return(betaSampled)
}

#############################################################
# Function to sample residual variance values using Metropolis Hastings  #
#############################################################

sampleSigeMetropolis<-function(sigeCurrent,y,X,betaCurrent){
  #using a inverted X^2 distribution for proposal
  #calculate residuals using the current value for beta
  res=y-X%*%betaCurrent
  #putting the current value for the residual variance into the fully conditional distribution
  pCurrentConditional=sigeCurrent**(-1*(length(y))/2)*exp(-.5*t(res)%*%res*(1/sigeCurrent))

  #putting the current value for the residual variance into the density of the proposal distribution
  pCurrentProposal=dchisq((sigeCurrent)/t(res)%*%res,0,length(res)- 3, log=FALSE)
  i=0

  while(i<1){
    #sample from proposal - inverted chisquare with d=n-3 and S=SSE
    sigeProposed=t(res)%*%res*(1/rchisq(1,length(res) - 3))

    #calculate MH ratio
    #putting the proposed value for the residual variance into the fully conditional distribution
    pProposedConditional=sigeProposed**(-1*(length(y))/2)*exp(-.5*t(res)%*%res*(1/sigeProposed))

    #putting the proposed value for the residual variance into the density of the proposal distribution
    pProposedProposal=dchisq((sigeProposed)/t(res)%*%res, 0,length(res)- 3, log=FALSE)

     #calculating Metropolis Hastings ratio


    MHr= #your code here


    #if the denominator is very small (zero) the ratio is NA
    if(!is.na(MHr)){
      #if the ratio is greater than 1 we automatically accept
      if(MHr>=1){
        sigeSampled=sigeProposed
        i=1
      } else {
        #if the ratio is less than one we sample a random uniform variable between 0 and 1
        randomNumber=runif(1,0,1)
        #if the sampled number is less than the ratio we accept
        if(randomNumber<=MHr){
          sigeSampled=sigeProposed
          i=1
        }
      }
    } else {
      #sigeSampled=NA
      sigeSampled=sigeProposed
      i=1
    }
  }

  return(sigeSampled[1,1])
}



### Running the M-H algorith and printing the results

The first thing to check when running a Bayesian analysis is the mixing of the Markov Chain. We do this by plotting samples in the order in which they were sampled. We do this to confirm that the Markov Chain has "burned in" to the target distribution and that the samples are independent (i.e. no auto-correlation)

In [ ]:
# solve M-H
#samples stores the results from each sample from mcmc
samples=solveBayesMetropolis(1000,X,y,c(mean(y[1:30]),mean(y[31:60]),mean(y[61:90])),var(y))

plot(samples[,1], main = "Samples for Beta 1")
plot(samples[,2], main = "Samples for Beta 2")
plot(samples[,3], main = "Samples for Beta 3")
plot(samples[,4], main = "Samples for the residual variance")


Next we look at the distribution of each parameter:

In [ ]:
hist(samples[300:length(samples[,1]),1], main = "Samples for Beta 1")
hist(samples[300:length(samples[,1]),2], main = "Samples for Beta 2")
hist(samples[300:length(samples[,1]),3], main = "Samples for Beta 3")
hist(samples[300:length(samples[,1]),4], main = "Samples for the residual variance")

Finally we can calculate some summary statistics

In [ ]:
nSamp=length(samples[,1])
print("Mean of Beta 1 samples")
print(mean(samples[300:nSamp,1]))
print("Mean of Beta 2 samples")
print(mean(samples[300:nSamp,2]))
print("Mean of Beta 3 samples")
print(mean(samples[300:nSamp,3]))
print("Mean of residual variance samples")
print(mean(samples[300:nSamp,4]))
print("Mode of residual variance samples")
density(samples[300:nSamp,4])$x[which.max(density(samples[300:nSamp,4])$y)]

## Gibbs Sampler

Gibbs is a special case of the Metroplis Hastings algorithm in which the proposal distribution is the fully conditional distribution. When this is the case the acceptance probability is always 1.

In [ ]:
solveBayesGibbs<-function(nIter,X,y,initialSige){

  #initializing the error variance
  currentSige=initialSige
  nrecords=length(y)
  #initializing matrix to store beta results
  #col 1 = beta1, col2 = beta2, col3 = beta3, col4 = error variance
  samples=matrix(0,nIter,4)
  for(i in c(1:nIter)){
    #print(i)
    #sample beta

    #using uninformative prior beta uniform (-infinity to positive infinity)
    currentBeta=mvrnorm(1,solve(t(X)%*%X)%*%t(X)%*%y,solve(t(X)%*%X)*currentSige)

    #using uninformative prior inverse chi-square(d=-3,S=0)
    sampleSige=t(y-X%*%currentBeta)%*%(y-X%*%currentBeta)*(1/rchisq(1,nrecords - 3))

    #updating the samples
    currentSige=sampleSige[1,1]
    samples[i,1:3]=currentBeta
    samples[i,4]=currentSige

  }

  return(samples)
}


Running the Gibbs Sampler and checking the mixing of the chain

In [ ]:
#samples stores the results from each sample from mcmc
samplesG = solveBayesGibbs(1000,X,y,10000)

plot(samplesG[,1], main = "Samples for Beta 1")
plot(samplesG[,2], main = "Samples for Beta 2")
plot(samplesG[,3], main = "Samples for Beta 3")
plot(samplesG[,4], main = "Samples for the residual variance")


looking at the distribution of each parameter:

In [ ]:
hist(samplesG[,1], main = "Samples for Beta 1")
hist(samplesG[,2], main = "Samples for Beta 2")
hist(samplesG[,3], main = "Samples for Beta 3")
hist(samplesG[,4], main = "Samples for the residual variance")

Finally we can calculate some summary statistics

In [ ]:
nSamp=length(samplesG[,1])
print("Mean of Beta 1 samples")
print(mean(samplesG[300:nSamp,1]))
print("Mean of Beta 2 samples")
print(mean(samplesG[300:nSamp,2]))
print("Mean of Beta 3 samples")
print(mean(samplesG[300:nSamp,3]))
print("Mean of residual variance samples")
print(mean(samplesG[300:nSamp,4]))
print("Mode of residual variance samples")
density(samplesG[300:nSamp,4])$x[which.max(density(samplesG[300:nSamp,4])$y)]

## Questions:

- **Complete the code for he M-H algorithm and confirm it is properly implementing MCMC (6 pts)**
- **Is there a difference in the performance (run time) of the M-H and Gibbs samplers? Why? (2 pts)**
- **Is the mean or mode from the MCMC residual variance samples closer to the LSM estimate of the residual variance? Why? (2 pts)**
